In [ ]:
import json
import javalang
from typing import Dict, List, Optional
import re
from transformers import AutoTokenizer
import torch

TOKENIZER = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")

def clean_code(code: str) -> str:
    """Clean Java code snippet by removing markdown, normalizing whitespace, and standardizing indentation."""
    code = re.sub(r'```java\n|```', '', code).strip()
    lines = code.split('\n')
    cleaned_lines = [line.rstrip() for line in lines if line.strip()]
    indented_lines = ['    ' + line.lstrip() if line.strip() else line for line in cleaned_lines]
    return '\n'.join(indented_lines)

def parse_java_code(code: str) -> Dict:
    """Parse Java code using javalang to extract class/method signatures, control structures, and sensitive APIs."""
    try:
        tree = javalang.parse.parse(code)
        metadata = {
            "classes": [],
            "methods": [],
            "control_structures": [],
            "sensitive_api_calls": [],
            "variables": []
        }
        for path, node in tree:
            if isinstance(node, javalang.tree.ClassDeclaration):
                metadata["classes"].append(node.name)
            elif isinstance(node, javalang.tree.MethodDeclaration):
                metadata["methods"].append({
                    "name": node.name,
                    "parameters": [param.name for param in node.parameters],
                    "return_type": node.return_type.name if node.return_type else "void"
                })
            elif isinstance(node, javalang.tree.IfStatement):
                metadata["control_structures"].append("if")
            elif isinstance(node, javalang.tree.ForStatement):
                metadata["control_structures"].append("for")
            elif isinstance(node, javalang.tree.WhileStatement):
                metadata["control_structures"].append("while")
            elif isinstance(node, javalang.tree.MethodInvocation):
                sensitive_apis = ['System.out.println', 'DriverManager.getConnection', 'ThreadLocal.get', 'Runtime.exec']
                if node.member in sensitive_apis:
                    metadata["sensitive_api_calls"].append(node.member)
            elif isinstance(node, javalang.tree.VariableDeclaration):
                var_type = node.type.name if node.type else "unknown"
                for declarator in node.declarators:
                    metadata["variables"].append({
                        "name": declarator.name,
                        "type": var_type
                    })
        return metadata
    except (javalang.parser.JavaSyntaxError, javalang.tokenizer.LexerError) as e:
        print(f"Parsing error ({type(e).__name__}) during parsing: {e}. Code snippet: \n{code[:200]}...")
        raise  
    except TypeError as e:
        print(f"TypeError during parsing: {e}. Code snippet: \n{code[:200]}...")
        raise  

def segment_code(code: str, max_tokens: int = 128000) -> List[str]:
    """Segment code into method-level units, ensuring each fits within token limits."""
    try:
        tree = javalang.parse.parse(code)
        methods = []
        for path, node in tree.filter(javalang.tree.MethodDeclaration):
            method_code = f"{node.return_type.name if node.return_type else 'void'} {node.name}() {{ {str(node.body) if node.body else ''} }}"
            token_count = len(TOKENIZER.encode(method_code))
            if token_count <= max_tokens:
                methods.append(method_code)
        if not methods:
            token_count = len(TOKENIZER.encode(code))
            if token_count <= max_tokens:
                methods = [code]
            else:
                methods = [code[:TOKENIZER.model_max_length]]  
        return methods
    except (javalang.parser.JavaSyntaxError, javalang.tokenizer.LexerError) as e:
        print(f"Parsing error ({type(e).__name__}) during segmentation: {e}. Code snippet: \n{code[:200]}...")
        token_count = len(TOKENIZER.encode(code))
        if token_count <= max_tokens:
            return [code]
        return [code[:TOKENIZER.model_max_length]]
    except TypeError as e:
        print(f"TypeError during segmentation: {e}. Code snippet: \n{code[:200]}...")
        raise  

def simple_data_flow_analysis(code: str) -> Dict:
    """Perform basic data flow analysis to identify taint sources and sinks."""
    taint_sources = []
    taint_sinks = []
    source_patterns = [r'\bSystem\.in\b', r'\bgetParameter\b']
    sink_patterns = [r'\bexecuteQuery\b', r'\bSystem\.out\.println\b', r'\bgetConnection\b', r'\bRuntime\.exec\b']
    for pattern in source_patterns:
        if re.search(pattern, code):
            taint_sources.append(pattern)
    for pattern in sink_patterns:
        if re.search(pattern, code):
            taint_sinks.append(pattern)
    return {"taint_sources": taint_sources, "taint_sinks": taint_sinks}

def generate_natural_language_hint(vulnerability_type: str, classification: str) -> str:
    """Generate a natural language hint to guide LLM detection, handling diverse vulnerability types."""
    cwe_map = {
        "safe": "No CWE",
        "Cross-Site Scripting (XSS)": "CWE-79",
        "SQL Injection": "CWE-89",
        "Command Injection": "CWE-78",
        "Path Traversal": "CWE-22",
        "Buffer Overflow": "CWE-119",
        "Insecure Deserialization": "CWE-502",
        "NVD-CWE-noinfo": "Unknown CWE",
        "CWE-79": "CWE-79",
        "CWE-611": "CWE-611",
        "CWE-20": "CWE-20",
        "Improper Control of Generation of Code (“Code Injection”)": "CWE-94",
        "Improper Input Validation": "CWE-20",
        "Out-of-bounds Write": "CWE-787",
        "CWE-22": "CWE-22",
        "Deserialization of Untrusted Data": "CWE-502",
        "CWE-264": "CWE-264",
        "CWE-502: Deserialization of Untrusted Data": "CWE-502",
        "CWE-200": "CWE-200",
        "CWE-502": "CWE-502",
        "NVD-CWE-Other": "Unknown CWE",
        "CWE-352": "CWE-352",
        "CWE-863": "CWE-863",
        "NULL Pointer Dereference": "CWE-476",
        "CWE-532: Insertion of Sensitive Information into Log File": "CWE-532",
        "CWE-287": "CWE-287",
        "CWE-22: Improper Limitation of a Pathname to a Restricted Directory": "CWE-22",
        "CWE-918: Server-Side Request Forgery (SSRF)": "CWE-918",
        "CWE-400": "CWE-400",
        "CWE-89: SQL Injection": "CWE-89",
        "CWE-327: Use of a Broken or Risky Cryptographic Algorithm": "CWE-327",
        "CWE-611: Improper Restriction of XML External Entity Reference": "CWE-611",
        "CWE-74": "CWE-74",
        "CWE-835": "CWE-835",
        "CWE-384: Session Fixation": "CWE-384",
        "CWE-330: Use of Insufficiently Random Values": "CWE-330",
        "CWE-94": "CWE-94",
        "CWE-444": "CWE-444",
        "CWE-284": "CWE-284",
        "Improper Neutralization of Special Elements used in an SQL Command (“SQL Injection”)": "CWE-89",
        "CWE-78: OS Command Injection": "CWE-78",
        "CWE-377: Insecure Temporary File": "CWE-377",
        "Improper Output Neutralization for Logs": "CWE-117",
        "CWE-285: Improper Authorization": "CWE-285",
        "CWE-79: Cross-Site Scripting (XSS)": "CWE-79",
        "CWE-295": "CWE-295",
        "CWE-434: Unrestricted Upload of File with Dangerous Type": "CWE-434",
        "CWE-755": "CWE-755",
        "CWE-89": "CWE-89",
        "CWE-352: Cross-Site Request Forgery (CSRF)": "CWE-352",
        "CWE-798: Use of Hard-coded Credentials": "CWE-798",
        "CWE-400: Uncontrolled Resource Consumption": "CWE-400",
        "CWE-321: Use of Hard-coded Cryptographic Key": "CWE-321",
        "CWE-862": "CWE-862",
        "CWE-601: URL Redirection to Untrusted Site ('Open Redirect')": "CWE-601",
        "CWE-295: Improper Certificate Validation": "CWE-295",
        "CWE-78": "CWE-78",
        "CWE-532": "CWE-532",
        "CWE-778: Insufficient Logging": "CWE-778",
        "CWE-399": "CWE-399",
        "CWE-327": "CWE-327",
        "CWE-732": "CWE-732",
        "CWE-347": "CWE-347",
        "CWE-770": "CWE-770",
        "CWE-190: Integer Overflow or Wraparound": "CWE-190",
        "CWE-269": "CWE-269",
        "CWE-1333: Inefficient Regular Expression Complexity": "CWE-1333",
        "CWE-209: Information Exposure Through an Error Message": "CWE-209",
        "CWE-306: Missing Authentication for Critical Function": "CWE-306",
        "CWE-378": "CWE-378",
        "CWE-369: Divide By Zero": "CWE-369",
        "CWE-470: Use of Externally-Controlled Input to Select Classes or Code": "CWE-470",
        "CWE-203": "CWE-203",
        "CWE-22: Improper Limitation of a Pathname to a Restricted Directory ('Path Traversal')": "CWE-22",
        "CWE-404: Improper Resource Shutdown or Release": "CWE-404",
        "CWE-564: SQL Injection in HQL (Hibernate)": "CWE-89",
        "CWE-362": "CWE-362",
        "CWE-329: Not Using a Random IV with CBC Mode": "CWE-329",
        "CWE-316": "CWE-316",
        "CWE-667: Improper Locking": "CWE-667",
        "CWE-614: Sensitive Cookie in HTTPS Session Without 'Secure' Attribute": "CWE-614",
        "CWE-601": "CWE-601",
        "CWE-284: Improper Access Control": "CWE-284",
        "CWE-116": "CWE-116",
        "CWE-328: Use of Weak Hash": "CWE-328",
        "CWE-524: Information Exposure Through Debug Information": "CWE-524",
        "CWE-19": "CWE-19",
        "CWE-862: Missing Authorization": "CWE-862",
        "CWE-90: LDAP Injection": "CWE-90",
        "CWE-59": "CWE-59",
        "CWE-367: Time-of-check Time-of-use (TOCTOU) Race Condition": "CWE-367",
        "CWE-459: Incomplete Cleanup": "CWE-459",
        "CWE-476: NULL Pointer Dereference": "CWE-476",
        "CWE-287: Improper Authentication": "CWE-287",
        "CWE-601: URL Redirection to Untrusted Site": "CWE-601",
        "CWE-790": "CWE-790",
        "CWE-276: Incorrect Default Permissions": "CWE-276",
        "CWE-409: Improper Handling of Highly Compressed Data (Zip Bomb)": "CWE-409",
        "CWE-470: Use of Externally-Controlled Input to Select Classes or Code ('Unsafe Reflection')": "CWE-470",
        "CWE-79: Cross-site Scripting": "CWE-79",
        "CWE-843": "CWE-843",
        "CWE-668": "CWE-668",
        "CWE-926": "CWE-926",
        "CWE-256: Plaintext Storage of a Password": "CWE-256",
        "CWE-252: Unchecked Return Value": "CWE-252",
        "CWE-434": "CWE-434",
        "CWE-1021": "CWE-1021",
        "CWE-1021: Improper Restriction of Rendered UI Layers or Frames": "CWE-1021",
        "CWE-208: Observable Timing Discrepancy": "CWE-208",
        "CWE-15: External Control of System or Configuration Setting": "CWE-15",
        "CWE-117: Improper Output Neutralization for Logs": "CWE-117",
        "CWE-613: Insufficient Session Expiration": "CWE-613",
        "CWE-16": "CWE-16",
        "CWE-264: Permissions, Privileges, and Access Controls": "CWE-264",
        "CWE-248: Reliance on Exception Handling in Security-Critical Code": "CWE-248",
        "CWE-78: Improper Neutralization of Special Elements used in an OS Command ('OS Command Injection')": "CWE-78",
        "CWE-89: Improper Neutralization of Special Elements used in an SQL Command ('SQL Injection')": "CWE-89",
        "CWE-94: Improper Control of Generation of Code ('Code Injection')": "CWE-94",
        "CWE-200: Information Exposure": "CWE-200",
        "CWE-835: Loop with Unreachable Exit Condition": "CWE-835",
        "CWE-113: HTTP Response Splitting": "CWE-113",
        "CWE-639: Authorization Bypass Through User-Controlled Key": "CWE-639",
        "CWE-312: Cleartext Storage of Sensitive Information": "CWE-312",
        "CWE-427: Uncontrolled Search Path Element": "CWE-427",
        "CWE-625: Permissive Regular Expression": "CWE-625",
        "CWE-200: Exposure of Sensitive Information to an Unauthorized Actor": "CWE-200",
        "Improper Neutralization of Input During Web Page Generation (“Cross-site Scripting”)": "CWE-79",
        "CWE-489: Active Debug Code": "CWE-489",
        "CWE-259: Use of Hard-coded Password": "CWE-259",
        "CWE-191": "CWE-191",
        "CWE-235": "CWE-235",
        "CWE-194": "CWE-194",
        "CWE-509": "CWE-509",
        "CWE-388": "CWE-388",
        "CWE-79: Improper Neutralization of Input During Web Page Generation ('Cross-site Scripting')": "CWE-79",
        "CWE-320": "CWE-320",
        "CWE-470": "CWE-470",
        "CWE-776": "CWE-776",
        "CWE-22: Improper Limitation of a Pathname to a Restricted Directory ('Zip Slip')": "CWE-22",
        "CWE-1004: Sensitive Cookie Without 'HttpOnly' Flag": "CWE-1004",
        "CWE-917: Improper Neutralization of Special Elements used in an Expression Language Statement": "CWE-917",
        "CWE-605: Multiple Binds to the Same Port (Hard-Coded Port)": "CWE-605",
        "CWE-552": "CWE-552",
        "CWE-311: Missing Encryption of Sensitive Data": "CWE-311",
        "CWE-491: Public cloneable() Method Without Final": "CWE-491",
        "CWE-639": "CWE-639",
        "CWE-337: Predictable Seed in Pseudo-Random Number Generator": "CWE-337",
        "CWE-384": "CWE-384",
        "CWE-189": "CWE-189",
        "Improper Restriction of XML External Entity Reference": "CWE-611",
        "CWE-476": "CWE-476",
        "CWE-490: Incomplete Cleanup": "CWE-490",
        "CWE-307: Improper Restriction of Excessive Authentication Attempts": "CWE-307",
        "CWE-377": "CWE-377",
        "CWE-522": "CWE-522",
        "CWE-184": "CWE-184",
        "CWE-116: Improper Encoding or Escaping of Output": "CWE-116",
        "CWE-209": "CWE-209",
        "CWE-94: Code Injection": "CWE-94",
        "CWE-319: Cleartext Transmission of Sensitive Information": "CWE-319",
        "CWE-770: Allocation of Resources Without Limits or Throttling": "CWE-770",
        "CWE-369": "CWE-369",
        "CWE-306": "CWE-306",
        "CWE-379: Creation of Temporary File in Directory with Incorrect Permissions": "CWE-379",
        "CWE-22: Path Traversal": "CWE-22",
        "CWE-925": "CWE-925",
        "CWE-209: Generation of Error Message Containing Sensitive Information": "CWE-209",
        "CWE-345: Insufficient Verification of Data Authenticity": "CWE-345",
        "Insecure Deserialization in Java can lead to Remote Code Execution": "CWE-502",
        "CWE-80": "CWE-80",
        "CWE-359": "CWE-359",
        "CWE-732: Incorrect Permission Assignment for Critical Resource": "CWE-732",
        "CWE-495: Private Data Structure Returned From A Public Method": "CWE-495",
        "CWE-434: Unrestricted File Upload": "CWE-434",
        "CWE-521: Weak Password Requirements": "CWE-521",
        "CWE-770: Missing Rate Limiting": "CWE-770",
        "CWE-315: Cleartext Storage of Sensitive Information in a Cookie": "CWE-315",
        "CWE-586: Explicit Call to Finalize()": "CWE-586",
        "CWE-436": "CWE-436",
        "CWE-185: Incorrect Regular Expression": "CWE-185",
        "CWE-90: Improper Neutralization of Special Elements used in an LDAP Query": "CWE-90",
        "CWE-20: Improper Input Validation": "CWE-20",
        "CWE-497": "CWE-497",
        "CWE-939": "CWE-939",
        "CWE-89/SQL Injection": "CWE-89",
        "CWE-15": "CWE-15",
        "CWE-525: Sensitive Information Exposure Through Browser Caching": "CWE-525",
        "CWE-644: Improper Neutralization of HTTP Headers": "CWE-644",
        "Improper Neutralization of Special Elements used in an OS Command (“OS Command Injection”)": "CWE-78",
        "CWE-311": "CWE-311",
        "CWE-310": "CWE-310",
        "CWE-494: Download of Code Without Integrity Check": "CWE-494",
        "CWE-1188": "CWE-1188",
        "CWE-23: Relative Path Traversal": "CWE-23",
        "CWE-400: Uncontrolled Resource Consumption ('ReDoS')": "CWE-400",
        "CWE-523: Unprotected Transport of Credentials": "CWE-523",
        "CWE-1104: Use of Unmaintained Third-Party Components": "CWE-1104",
        "CWE-501": "CWE-501",
        "CWE-499: Serializable Class Containing Sensitive Data": "CWE-499",
        "CWE-776: Improper Handling of Highly Compressed Data (Zip Bomb)": "CWE-776",
        "CWE-316: Cleartext Storage of Sensitive Information in Memory": "CWE-316",
        "CWE-129: Improper Validation of Array Index": "CWE-129",
        "CWE-1333": "CWE-1333",
        "CWE-780": "CWE-780",
        "CWE-401": "CWE-401",
        "CWE-319": "CWE-319",
        "CWE-362: Concurrent Execution using Shared Resource with Improper Synchronization": "CWE-362",
        "CWE-208": "CWE-208",
        "CWE-759: Use of a One-Way Hash without a Salt": "CWE-759",
        "CWE-459: Incomplete Cleanup of Temporary Files": "CWE-459",
        "CWE-451": "CWE-451",
        "CWE-337": "CWE-337",
        "CWE-125": "CWE-125",
        "CWE-361": "CWE-361",
        "CWE-918": "CWE-918",
        "CWE-1104": "CWE-1104",
        "CWE-1004": "CWE-1004",
        "CWE-276": "CWE-276",
        "CWE-749": "CWE-749",
        "CWE-119": "CWE-119",
        "CWE-1041": "CWE-1041",
        "CWE-362: Concurrent Execution using Shared Resource with Improper Synchronization ('Race Condition')": "CWE-362",
        "CWE-787: Out-of-bounds Write": "CWE-787",
        "CWE-119: Improper Restriction of Operations within the Bounds of a Memory Buffer": "CWE-119",
        "CWE-416: Use After Free": "CWE-416",
        "CWE-1039": "CWE-1039",
        "CWE-244": "CWE-244",
        "Improper Input Validation in Java can lead to Remote Code Execution": "CWE-20",
        "CWE-244: Improper Clearing of Sensitive Data": "CWE-244",
        "CWE-778: Insufficient Logging & Monitoring": "CWE-778",
        "CWE-532: Information Exposure Through Log Files": "CWE-532",
        "CWE-749: Exposed Dangerous Method": "CWE-749",
        "CWE-459": "CWE-459",
        "CWE-338: Use of Cryptographically Weak Pseudo-Random Number Generator": "CWE-338",
        "CWE-315": "CWE-315",
        "CWE-95: Improper Neutralization of Directives in Dynamically Evaluated Code": "CWE-95",
        "CWE-643: Improper Handling of XPath Expression": "CWE-643",
        "CWE-345": "CWE-345",
        "CWE-754": "CWE-754",
        "CWE-829: Inclusion of Functionality from Untrusted Control Sphere": "CWE-829",
        "CWE-337: Predictable Seed in PRNG": "CWE-337",
        "CWE-331: Insufficient Entropy": "CWE-331",
        "CWE-780: Use of RSA Algorithm without OAEP": "CWE-780",
        "CWE-338: Use of Cryptographically Weak PRNG": "CWE-338",
        "CWE-1333: Inefficient Regular Expression Complexity (ReDoS)": "CWE-1333",
        "CWE-401: Improper Release of Memory Before Removing Last Reference": "CWE-401",
        "CWE-328: Reversible One-Way Hash": "CWE-328",
        "CWE-426: Untrusted Search Path": "CWE-426",
        "CWE-522: Insufficiently Protected Credentials": "CWE-522",
        "CWE-1035": "CWE-1035",
        "CWE-79: Cross-site Scripting (XSS)": "CWE-79",
        "CWE-79/Cross-Site Scripting": "CWE-79",
        "CWE-502/Deserialization of Untrusted Data": "CWE-502",
        "CWE-326: Inadequate Encryption Strength": "CWE-326",
        "CWE-591": "CWE-591",
        "CWE-78: Improper Neutralization of Special Elements used in an OS Command": "CWE-78",
        "CWE-338: Use of Cryptographically Weak Pseudo-Random Number Generator (PRNG)": "CWE-338",
        "CWE-340: Generation of Predictable Numbers or Identifiers": "CWE-340",
        "CWE-497: Exposure of Sensitive System Information to an Unauthorized Control Sphere": "CWE-497",
        "CWE-532/Insertion of Sensitive Information into Log File": "CWE-532",
        "CWE-328": "CWE-328",
        "CWE-409: Zip Bomb Attack": "CWE-409",
        "CWE-327: Use of a Broken or Risky Cryptographic Algorithm (MD5)": "CWE-327",
        "CWE-829": "CWE-829",
        "CWE-79: Improper Neutralization of Input During Web Page Generation": "CWE-79",
        "CWE-400: Regular Expression DoS (ReDoS)": "CWE-400",
        "CWE-843: Access of Resource Using Incompatible Type": "CWE-843",
        "CWE-776: XML Entity Expansion": "CWE-776",
        "CWE-209: Information Exposure Through Error Messages": "CWE-209",
        "CWE-620: Unverified State Change": "CWE-620",
        "CWE-611: XML External Entity (XXE)": "CWE-611",
        "Inadequate input validation and sanitization can lead to SQL Injection in Java applications": "CWE-89",
        "The use of Thread.stop() method in Java can lead to unpredictable behavior and data corruption": "CWE-691",
        "CWE-502: Unsafe Deserialization": "CWE-502",
        "CWE-287: Broken Authentication": "CWE-287",
        "Insecure deserialization of objects can lead to Remote Code Execution": "CWE-502",
        "Inadequate use of input validation and sanitization can lead to SQL Injection vulnerabilities": "CWE-89",
        "Insecure Deserialization in Java can lead to Remote Code Execution, allowing attackers to execute arbitrary code on the system": "CWE-502",
        "CWE-259: Hard-coded Password": "CWE-259",
        "CWE-664: Improper Control of a Resource Through its Lifetime": "CWE-664",
        "CWE-208: Timing Attack": "CWE-208",
        "CWE-335": "CWE-335",
        "CWE-404": "CWE-404",
        "Insecure deserialization of objects can lead to Remote Code Execution": "CWE-502",
        "CWE-327: Use of Broken Crypto (DES)": "CWE-327",
        "CWE-307: Improper Restriction of Auth Attempts": "CWE-307",
        "CWE-918: Server-Side Request Forgery": "CWE-918",
        "CWE-352: Missing CSRF Protection": "CWE-352",
        "CWE-532: Info Exposure Through Logs": "CWE-532",
        "CWE-470: Reflection Injection": "CWE-470",
        "CWE-321: Hardcoded Cryptographic Key": "CWE-321",
        "CWE-330: Insufficient Entropy in PRNG": "CWE-330",
        "CWE-304: Missing Critical Step in Auth": "CWE-304",
        "CWE-921: Storage of Sensitive Data in Mechanism Without Access Control": "CWE-921",
        "Insecure deserialization of objects in Java can lead to Remote Code Execution": "CWE-502",
        "Java's Thread.stop() method can lead to unpredictable behavior and data corruption": "CWE-691",
        "The use of user-supplied input without proper sanitization can lead to SQL Injection vulnerabilities": "CWE-89",
        "CWE-384/Session Fixation": "CWE-384",
        "CWE-798/Use of Hard-coded Credentials": "CWE-798",
        "CWE-327/Use of a Broken or Risky Cryptographic Algorithm": "CWE-327",
        "CWE-601/URL Redirection to Untrusted Site": "CWE-601",
        "CWE-1333/Inefficient Regular Expression Complexity": "CWE-1333",
        "CWE-209: Information Exposure Through Error Message": "CWE-209",
        "CWE-400: Uncontrolled Resource Consumption ('Regular Expression DoS')": "CWE-400",
        "CWE-304": "CWE-304",
        "CWE-96: Improper Neutralization of Directives in Statically Stored Code ('Static Code Injection')": "CWE-96",
        "CWE-611/XML External Entity Injection": "CWE-611",
        "CWE-640": "CWE-640",
        "CWE-240": "CWE-240",
        "CWE-337: Use of Less-Random Values in sensitive contexts": "CWE-337",
        "CWE-95: Improper Neutralization of Directives in Dynamically Evaluated Code ('Eval Injection')": "CWE-95",
        "CWE-863: Incorrect Authorization": "CWE-863",
        "CWE-502: Insecure Deserialization": "CWE-502",
        "CWE-610": "CWE-610",
        "CWE-287/Improper Authentication": "CWE-287",
        "CWE-77": "CWE-77",
        "CWE-538": "CWE-538",
        "CWE-338": "CWE-338",
        "CWE-425: Direct Request ('Forced Browsing')": "CWE-425",
        "CWE-613": "CWE-613",
        "CWE-113": "CWE-113",
        "CWE-22/Path Traversal": "CWE-22",
        "CWE-117/Improper Output Neutralization for Logs": "CWE-117",
        "CWE-17": "CWE-17",
        "CWE-90/LDAP Injection": "CWE-90",
        "CWE-223/Omission of Security-relevant Information": "CWE-223",
        "CWE-125: Out-of-bounds Read": "CWE-125",
        "CWE-611: XML External Entity (XXE) Injection": "CWE-611",
        "CWE-835: Loop with Unreachable Exit Condition ('Infinite Loop')": "CWE-835",
        "CWE-435": "CWE-435",
        "CWE-337: Use of Predictable Algorithm in Random Number Generator": "CWE-337",
        "CWE-362: Race Condition": "CWE-362",
        "CWE-451: User Interface (UI) Misrepresentation of Critical Information": "CWE-451",
        "CWE-354: Improper Validation of Integrity Check Value": "CWE-354",
        "CWE-428": "CWE-428",
        "CWE-1039: Improper Implementation of SSL/TLS": "CWE-1039",
        "CWE-1041: Use of Predictable Initialization Vector": "CWE-1041",
        "CWE-90": "CWE-90",
        "CWE-471: Modification of Assumed-Immutable Data": "CWE-471",
        "CWE-22: Path Traversal (Zip Slip)": "CWE-22",
        "CWE-787": "CWE-787",
        "CWE-326": "CWE-326",
        "CWE-91": "CWE-91",
        "CWE-89: Improper Neutralization of Special Elements used in an SQL Command": "CWE-89",
        "CWE-917: Improper Neutralization of Special Elements used in an Expression Language (EL) Statement ('Expression Language Injection')": "CWE-917",
        "CWE-113: Improper Neutralization of Line Termination Characters": "CWE-113",
        "CWE-288: Authorization Bypass Through Relay": "CWE-288",
        "CWE-401: Improper Release of Memory Before Removing Last Reference ('Memory Leak')": "CWE-401",
        "CWE-79: Improper Neutralization of Input During Web Page Generation ('XSS')": "CWE-79",
        "CWE-208: Observable Response Discrepancy (Timing Attack)": "CWE-208",
        "CWE-297: Improper Validation of Certificate with Host Mismatch": "CWE-297",
        "CWE-611: Improper Restriction of XML External Entity Reference ('XXE')": "CWE-611",
        "CWE-328: Use of Weak Hash (SHA-1) in Digital Signature": "CWE-328",
        "CWE-400: Uncontrolled Resource Consumption (Thread Exhaustion)": "CWE-400",
        "CWE-327: Use of a Broken or Risky Cryptographic Algorithm (DES)": "CWE-327",
        "CWE-328: Use of Weak Hash (MD5)": "CWE-328",
        "CWE-327: Use of Broken or Risky Cryptographic Algorithm": "CWE-327",
        "CWE-564: SQL Injection in HQL": "CWE-89",
        "CWE-352: Cross-Site Request Forgery": "CWE-352",
        "Insecure deserialization of objects in Java can lead to Remote Code Execution": "CWE-502",
        "Integer Overflow or Wraparound": "CWE-190"
    }

    if vulnerability_type is None:
        cwe = "No CWE"
        vuln_desc = "no specific vulnerability"
    else:
        cwe = cwe_map.get(vulnerability_type, "Unknown CWE")
        vuln_desc = vulnerability_type.lower()

    if classification == "VULNERABLE":
        return f"Check for {vuln_desc} vulnerabilities ({cwe}), such as improper handling of user inputs, unsafe API usage, or concurrency issues."
    return f"Verify that the code follows secure coding practices and contains no {vuln_desc} vulnerabilities ({cwe})."

def process_jsonl_entry(entry: Dict) -> Dict:
    """Process a single JSONL entry according to the methodology, using original code on error."""
    code = entry["code"]
    vulnerability_type = entry["vulnerability_type"]
    classification = entry["classification"]

    try:
        cleaned_code = clean_code(code)
        parsed_metadata = parse_java_code(cleaned_code)
        if "error" in parsed_metadata:
            raise Exception(f"Parsing failed: {parsed_metadata['error']}")

        segmented_code = segment_code(cleaned_code)
        data_flow = simple_data_flow_analysis(cleaned_code)
        nl_hint = generate_natural_language_hint(vulnerability_type, classification)

        messages = json.dumps({
            "code": cleaned_code,
            "metadata": {
                **parsed_metadata,
                "data_flow": data_flow
            },
            "hint": nl_hint
        })
        code_content = messages
    except Exception as e:
        print(f"Error processing entry: {e}. Using original code.")
        code_content = code  

    prompt_content = {
        "code": code_content,
        "vulnerability_type": vulnerability_type,
        "classification": classification
    }
    return prompt_content

def main(jsonl_file: str, output_file: str):
    """Process JSONL dataset and save preprocessed data for LLM prompts."""
    processed_entries = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                entry = json.loads(line.strip())
                processed_entry = process_jsonl_entry(entry)
                if processed_entry:  
                    processed_entries.append(processed_entry)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}. Skipping line.")

    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in processed_entries:
            f.write(json.dumps(entry) + '\n')



In [ ]:
main("clean_merged.jsonl", "processed_datasetv1.jsonl")

In [ ]:
!pip install javalang

In [ ]:
import pandas as pd
df=pd.read_json("final_deduped.jsonl", lines=True)

In [ ]:
df["classification"].value_counts()

In [ ]:
df["vulnerability_type"].value_counts()

In [ ]:
with open('vulnerability_type.txt','w')as file_out:
  file_out.write(df["vulnerability_type"].value_counts().to_string())

In [ ]:
import pandas as pd
df=pd.read_json("./processed_datasetv1.jsonl", lines=True)

In [ ]:
df["classification"].value_counts()